## Transfer Learning usando dados MNIST
Para ilustrar o poder e o conceito de transfer learning, treinaremos uma CNN apenas nos dígitos 5,6,7,8,9. Em seguida, treinaremos apenas a(s) última(s) camada(s) da rede nos dígitos 0,1,2,3,4 e veremos quão bem as características aprendidas nos dígitos 5-9 ajudam na classificação dos dígitos 0-4.

Adaptado do Exemplo do Keras.

In [26]:
# Preliminares

from __future__ import absolute_import, division, print_function  # Compatibilidade Python 2/3

import warnings
warnings.filterwarnings("ignore")

import datetime
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [27]:
## Importar objetos do Keras para Deep Learning

import tensorflow.keras as keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

In [28]:
# usado para ajudar algumas das funções de tempo
now = datetime.datetime.now

In [29]:
# definir alguns parâmetros
batch_size = 64
num_classes = 5
epochs = 5

In [30]:
# definir mais alguns parâmetros
img_rows, img_cols = 28, 28
filters = 32
pool_size = 2
kernel_size = 3

In [31]:
## Isso apenas lida com alguma variabilidade na forma como os dados de entrada são carregados

if K.image_data_format() == 'channels_first':
    input_shape = (1, img_rows, img_cols)
else:
    input_shape = (img_rows, img_cols, 1)

In [32]:
## Para simplificar as coisas, escreva uma função para incluir todas as etapas de treinamento
## Como entrada, a função recebe um modelo, conjunto de treinamento, conjunto de teste e o número de classes
## Dentro do objeto modelo estará o estado sobre quais camadas estamos congelando e quais estamos treinando

def train_model(model, train, test, num_classes):
    x_train = train[0].reshape((train[0].shape[0],) + input_shape)
    x_test = test[0].reshape((test[0].shape[0],) + input_shape)
    x_train = x_train.astype('float32')
    x_test = x_test.astype('float32')
    x_train /= 255
    x_test /= 255
    print('formato de x_train:', x_train.shape)
    print(x_train.shape[0], 'amostras de treino')
    print(x_test.shape[0], 'amostras de teste')

    # converter vetores de classe em matrizes de classe binárias
    y_train = keras.utils.to_categorical(train[1], num_classes)
    y_test = keras.utils.to_categorical(test[1], num_classes)

    model.compile(loss='categorical_crossentropy',
                  optimizer='adadelta',
                  metrics=['accuracy'])

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    t = now()
    model.fit(x_train, y_train,
              batch_size=batch_size,
              epochs=epochs,
              verbose=1,
              validation_data=(x_test, y_test),
              fallbacks=[early_stop])
    print('Tempo de treinamento: %s' % (now() - t))

    score = model.evaluate(x_test, y_test, verbose=0)
    print('Pontuação no teste:', score[0])
    print('Precisão no teste:', score[1])

In [33]:
# os dados, embaralhados e divididos entre conjuntos de treino e teste
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# criar dois conjuntos de dados: um com dígitos abaixo de 5 e outro com 5 e acima
x_train_lt5 = x_train[y_train < 5]
y_train_lt5 = y_train[y_train < 5]
x_test_lt5 = x_test[y_test < 5]
y_test_lt5 = y_test[y_test < 5]

x_train_gte5 = x_train[y_train >= 5]
y_train_gte5 = y_train[y_train >= 5] - 5
x_test_gte5 = x_test[y_test >= 5]
y_test_gte5 = y_test[y_test >= 5] - 5

In [34]:
# Definir as camadas de "características". Essas são as camadas iniciais que esperamos que "transfiram"
# para um novo problema. Congelaremos essas camadas durante o processo de fine-tuning

feature_layers = [
    Conv2D(filters, kernel_size,
           padding='valid',
           input_shape=input_shape),
    Activation('relu'),
    Conv2D(filters, kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size=pool_size),
    Dropout(0.25),
    Flatten(),
]

In [35]:
# Definir as camadas de "classificação". Essas são as camadas posteriores que predizem as classes específicas a partir das características
# aprendidas pelas camadas de características. Esta é a parte do modelo que precisa ser re-treinada para um novo problema

classification_layers = [
    Dense(128),
    Activation('relu'),
    Dropout(0.5),
    Dense(num_classes),
    Activation('softmax')
]

In [36]:
# Criamos nosso modelo combinando os dois conjuntos de camadas da seguinte forma
model = Sequential(feature_layers + classification_layers)

In [37]:
# Vamos dar uma olhada
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 5)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600,165 (2.29 MB)

 Trainable params: 600,165 (2.29 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# Agora, vamos treinar nosso modelo nos dígitos 5,6,7,8,9

train_model(model,
            (x_train_gte5, y_train_gte5),
            (x_test_gte5, y_test_gte5), num_classes)

formato de x_train: (29404, 28, 28, 1)
29404 amostras de treino
4861 amostras de teste
Epoch 1/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.7851 - loss: 0.7185 - val_accuracy: 0.8605 - val_loss: 0.5960
Epoch 2/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7991 - loss: 0.6902 - val_accuracy: 0.8685 - val_loss: 0.5734
Epoch 3/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8118 - loss: 0.6665 - val_accuracy: 0.8743 - val_loss: 0.5564
Epoch 4/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8136 - loss: 0.6505 - val_accuracy: 0.8766 - val_loss: 0.5416
Epoch 5/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8200 - loss: 0.6342 - val_accuracy: 0.8784 - val_loss: 0.5281
Tempo de treinamento: 0:00:12.422548
Pontuação no teste: 0.5280794501304626
Precisão no teste: 0.8784200549125671


o desempenho tava ruim, coloquei otimizador adam, diminui o batch size e coloquei early stop. diminui o batch size mais pra poder abaixar a lr do adam e combinar com earlystop

### Congelando Camadas
O Keras permite que as camadas sejam "congeladas" durante o processo de treinamento. Ou seja, algumas camadas teriam seus pesos atualizados durante o processo de treinamento, enquanto outras não. Esta é uma parte central do transfer learning, a capacidade de treinar apenas a última ou várias camadas.

Observe também que muito do tempo de treinamento é gasto "retropropagando" os gradientes de volta para a primeira camada. Portanto, se precisarmos apenas calcular os gradientes de volta a um pequeno número de camadas, o tempo de treinamento é muito mais rápido por iteração. Isso além das economias obtidas por poder treinar em um conjunto de dados menor.

In [38]:
# Congelar apenas as camadas de características
for l in feature_layers:
    l.trainable = False

Observe abaixo as diferenças entre o número de *parâmetros totais*, *parâmetros treináveis* e *parâmetros não-treináveis*.

In [39]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 5)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600,165 (2.29 MB)

 Trainable params: 590,597 (2.25 MB)

 Non-trainable params: 9,568 (37.38 KB)

In [40]:
train_model(model,
            (x_train_lt5, y_train_lt5),
            (x_test_lt5, y_test_lt5), num_classes)

formato de x_train: (30596, 28, 28, 1)
30596 amostras de treino
5139 amostras de teste
Epoch 1/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9550 - loss: 0.1412 - val_accuracy: 0.9879 - val_loss: 0.0367
Epoch 2/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9718 - loss: 0.0906 - val_accuracy: 0.9905 - val_loss: 0.0297
Epoch 3/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9763 - loss: 0.0748 - val_accuracy: 0.9912 - val_loss: 0.0240
Epoch 4/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9799 - loss: 0.0689 - val_accuracy: 0.9932 - val_loss: 0.0184
Epoch 5/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9772 - loss: 0.0722 - val_accuracy: 0.9922 - val_loss: 0.0228
Tempo de treinamento: 0:00:17.616172
Pontuação no teste: 0.02275599166750908
Precisão no teste: 0.9922164082527161


Observe que após uma única época, já estamos alcançando resultados na classificação de 0-4 que são comparáveis àqueles alcançados em 5-9 após 5 épocas completas. Isso apesar do fato de que estamos apenas "ajustando" a última camada da rede, e todas as camadas iniciais nunca viram como os dígitos 0-4 se parecem.

Além disso, observe que mesmo que quase todos (590K/600K) dos *parâmetros* fossem treináveis, o tempo de treinamento por época ainda foi muito reduzido. Isso ocorre porque a parte não congelada da rede era muito superficial, tornando a retropropagação mais rápida.

## Exercício
### Para fazer:
- Agora escreva código para reverter este processo de treinamento. Ou seja, você treinará nos dígitos 0-4, e então fará fine-tune apenas das últimas camadas nos dígitos 5-9.

In [46]:
# vou considerar que é pra usar a mesma arquitetura, e vou incluir os ajustes que eu fiz de otimização

img_rows, img_cols = 28, 28
filters = 32
pool_size = 2
kernel_size = 3
batch_size = 64
num_classes = 5
epochs = 5

feature_layers = [
    Conv2D(filters, kernel_size,
           padding='valid',
           input_shape=input_shape),
    Activation('relu'),
    Conv2D(filters, kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size=pool_size),
    Dropout(0.25),
    Flatten(),
]

classification_layers = [
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax'),
]

model2 = Sequential(feature_layers + classification_layers)
model2.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600,165 (2.29 MB)

 Trainable params: 600,165 (2.29 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
#Treinando globalmente com 0-4

train_model(model2, (x_train_lt5, y_train_lt5), (x_test_lt5, y_test_lt5), num_classes)

formato de x_train: (30596, 28, 28, 1)
30596 amostras de treino
5139 amostras de teste
Epoch 1/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9732 - loss: 0.0876 - val_accuracy: 0.9934 - val_loss: 0.0153
Epoch 2/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9879 - loss: 0.0405 - val_accuracy: 0.9975 - val_loss: 0.0079
Epoch 3/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9902 - loss: 0.0318 - val_accuracy: 0.9953 - val_loss: 0.0109
Epoch 4/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9903 - loss: 0.0330 - val_accuracy: 0.9971 - val_loss: 0.0068
Epoch 5/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9925 - loss: 0.0229 - val_accuracy: 0.9963 - val_loss: 0.0111
Tempo de treinamento: 0:00:19.695550
Pontuação no teste: 0.01105121523141861
Precisão no teste: 0.9963027834892273


In [45]:
for camada in feature_layers: camada.trainable = False
#Congelando camadas iniciais

model2.summary()

train_model(model2,(x_train_gte5, y_train_gte5),(x_test_gte5, y_test_gte5), num_classes)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,781,361 (6.80 MB)

 Trainable params: 590,597 (2.25 MB)

 Non-trainable params: 9,568 (37.38 KB)

 Optimizer params: 1,181,196 (4.51 MB)

formato de x_train: (29404, 28, 28, 1)
29404 amostras de treino
4861 amostras de teste
Epoch 1/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9867 - loss: 0.0420 - val_accuracy: 0.9914 - val_loss: 0.0351
Epoch 2/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9877 - loss: 0.0395 - val_accuracy: 0.9928 - val_loss: 0.0277
Epoch 3/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9903 - loss: 0.0325 - val_accuracy: 0.9932 - val_loss: 0.0288
Epoch 4/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9909 - loss: 0.0279 - val_accuracy: 0.9905 - val_loss: 0.0332
Epoch 5/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9908 - loss: 0.0309 - val_accuracy: 0.9922 - val_loss: 0.0298
Tempo de treinamento: 0:00:13.137900
Pontuação no teste: 0.029808523133397102
Precisão no teste: 0.9921826720237732


In [48]:
def train_model2(train1, test1, train2, test2, feature_layers, classification_layers, num_classes):
    x_train1 = train1[0].reshape((train1[0].shape[0],) + input_shape)
    x_test1 = test1[0].reshape((test1[0].shape[0],) + input_shape)
    x_train1 = x_train1.astype('float32')
    x_test1 = x_test1.astype('float32')
    x_train1 /= 255
    x_test1 /= 255
    print('formato de x_train - 1:', x_train1.shape)
    print(x_train1.shape[0], 'amostras de treino')
    print(x_test1.shape[0], 'amostras de teste')

    x_train2 = train2[0].reshape((train2[0].shape[0],) + input_shape)
    x_test2 = test2[0].reshape((test2[0].shape[0],) + input_shape)
    x_train2 = x_train2.astype('float32')
    x_test2 = x_test2.astype('float32')
    x_train2 /= 255
    x_test2 /= 255
    print('formato de x_train - 2:', x_train2.shape)
    print(x_train2.shape[0], 'amostras de treino')
    print(x_test2.shape[0], 'amostras de teste')

    # converter vetores de classe em matrizes de classe binárias
    y_train1 = keras.utils.to_categorical(train1[1], num_classes)
    y_test1 = keras.utils.to_categorical(test1[1], num_classes)
    y_train2 = keras.utils.to_categorical(train2[1], num_classes)
    y_test2 = keras.utils.to_categorical(test2[1], num_classes)

    model = Sequential(feature_layers + classification_layers)
    model.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.005),
                  metrics=['accuracy'])

    t = now()
    model.fit(x_train1, y_train1,
              batch_size=batch_size,
              epochs=epochs,
              verbose=1,
              validation_data=(x_test1, y_test1))
    print('Tempo de treinamento - 1: %s' % (now() - t))

    score1 = model.evaluate(x_test1, y_test1, verbose=0)
    print('Pontuação no teste:', score1[0])
    print('Precisão no teste:', score1[1])

    for camada in feature_layers: l.trainable = False

    t = now()
    model.fit(x_train2, y_train2,
              batch_size=batch_size,
              epochs=epochs,
              verbose=1,
              validation_data=(x_test2, y_test2))
    print('Tempo de treinamento - 2: %s' % (now() - t))

    score2 = model.evaluate(x_test2, y_test2, verbose=0)
    print('Pontuação no teste:', score2[0])
    print('Precisão no teste:', score2[1])

In [49]:
train_model2((x_train_lt5, y_train_lt5), (x_test_lt5, y_test_lt5),(x_train_gte5, y_train_gte5),(x_test_gte5, y_test_gte5),
              feature_layers, classification_layers, num_classes)

formato de x_train - 1: (30596, 28, 28, 1)
30596 amostras de treino
5139 amostras de teste
formato de x_train - 2: (29404, 28, 28, 1)
29404 amostras de treino
4861 amostras de teste
Epoch 1/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9699 - loss: 0.0963 - val_accuracy: 0.9944 - val_loss: 0.0177
Epoch 2/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9884 - loss: 0.0408 - val_accuracy: 0.9930 - val_loss: 0.0247
Epoch 3/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9887 - loss: 0.0377 - val_accuracy: 0.9965 - val_loss: 0.0106
Epoch 4/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9931 - loss: 0.0226 - val_accuracy: 0.9965 - val_loss: 0.0082
Epoch 5/5
479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9921 - loss: 0.0249 - val_accuracy: 0.9957 - val_loss: 0.0106
Tempo de treinamento - 1: 0:00:16.638466
Pontuação no teste: 0.010616828687489033
Precisão no teste: 0.9957190155982971
Epoch 1/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy

apareceu 1 milhao de pesos de otimizador, acho que é do adam, mas tirando isso, da pra ver 
os pesos congelando